# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


In [8]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()
print(df.columns.tolist())
df.head(3)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pagev

In [9]:
CLICK_PCT_GROW = 0.20      # +20% clicks, last 30d vs prev 30d
CLICK_PCT_DECLINE = -0.20  # -20% clicks
WEAK_POSITION_THRESHOLD = 10  # avg_position worse than top-10 = weak

REASON_CODES = {
    "CLICKS_DROP_SIG": "Clicks fell 20%+ vs. the prior 30-day period",
    "CLICKS_RISE_SIG": "Clicks rose 20%+ vs. the prior 30-day period",
    "WEAK_POSITION_FLAT": "Flat click trend, but ranking outside top 10",
    "NO_CLEAR_SIGNAL": "Flat click trend, ranking healthy — low urgency",
}

def click_pct_change(row):
    prev = row["clicks_prev_30d"]
    if prev == 0 or pd.isna(prev):
        return None  # can't compute a % change off a zero/missing base
    return (row["clicks_last_30d"] - prev) / prev

def score_row(row):
    pct_change = click_pct_change(row)
    if pct_change is None:
        return "review", "NO_CLEAR_SIGNAL", 0.0

    if pct_change >= CLICK_PCT_GROW:
        return "growing", "CLICKS_RISE_SIG", pct_change
    elif pct_change <= CLICK_PCT_DECLINE:
        return "declining", "CLICKS_DROP_SIG", pct_change
    elif row["avg_position"] > WEAK_POSITION_THRESHOLD:
        return "review", "WEAK_POSITION_FLAT", pct_change
    else:
        return "review", "NO_CLEAR_SIGNAL", pct_change

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import os
import pandas as pd

results = df.apply(score_row, axis=1, result_type="expand")
results.columns = ["action", "reason_code", "action_score"]

scored_df = pd.concat([df, results], axis=1)
scored_df["abs_action_score"] = scored_df["action_score"].abs()

ranked = scored_df.sort_values("abs_action_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
ranked[["rank", "content_id", "action", "reason_code", "action_score", "clicks_last_30d", "clicks_prev_30d", "avg_position"]].head(20)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,rank,content_id,action,reason_code,action_score,clicks_last_30d,clicks_prev_30d,avg_position
0,1,content_0e4ca10e0c65,growing,CLICKS_RISE_SIG,25.000000,26,1,7.3
1,2,content_69b2f71fccea,growing,CLICKS_RISE_SIG,22.000000,23,1,6.7
2,3,content_fac4a121be7e,growing,CLICKS_RISE_SIG,20.666667,65,3,26.7
3,4,content_fd2d2b57f310,growing,CLICKS_RISE_SIG,19.500000,41,2,16.4
4,5,content_78ede5574fe4,growing,CLICKS_RISE_SIG,18.000000,19,1,39.4
5,6,content_181d0c8829a8,growing,CLICKS_RISE_SIG,16.000000,17,1,13.3
6,7,content_5e69e1e71240,growing,CLICKS_RISE_SIG,16.000000,34,2,21.7
7,8,content_34e549c30fa0,growing,CLICKS_RISE_SIG,15.529412,562,34,6.3
8,9,content_2d665182534d,growing,CLICKS_RISE_SIG,15.000000,16,1,36.1
9,10,content_57ce8ef69462,growing,CLICKS_RISE_SIG,15.000000,16,1,27.8


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top20 = ranked.head(20)[[
    "rank", "content_id", "action", "reason_code", "action_score",
    "clicks_last_30d", "clicks_prev_30d", "avg_position", "days_with_impressions"
]]
top20


,rank,content_id,action,reason_code,action_score,clicks_last_30d,clicks_prev_30d,avg_position,days_with_impressions
0,1,content_0e4ca10e0c65,growing,CLICKS_RISE_SIG,25.000000,26,1,7.3,88
1,2,content_69b2f71fccea,growing,CLICKS_RISE_SIG,22.000000,23,1,6.7,88
2,3,content_fac4a121be7e,growing,CLICKS_RISE_SIG,20.666667,65,3,26.7,88
3,4,content_fd2d2b57f310,growing,CLICKS_RISE_SIG,19.500000,41,2,16.4,88
4,5,content_78ede5574fe4,growing,CLICKS_RISE_SIG,18.000000,19,1,39.4,88
5,6,content_181d0c8829a8,growing,CLICKS_RISE_SIG,16.000000,17,1,13.3,88
6,7,content_5e69e1e71240,growing,CLICKS_RISE_SIG,16.000000,34,2,21.7,88
7,8,content_34e549c30fa0,growing,CLICKS_RISE_SIG,15.529412,562,34,6.3,88
8,9,content_2d665182534d,growing,CLICKS_RISE_SIG,15.000000,16,1,36.1,88
9,10,content_57ce8ef69462,growing,CLICKS_RISE_SIG,15.000000,16,1,27.8,88


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# Weak picks: big % swing off a tiny click base — statistically fragile
weak_picks = ranked[
    (ranked["clicks_prev_30d"] < 10) & (ranked["abs_action_score"] > 0.5)
]
print(f"{len(weak_picks)} picks flagged as weak (swing off a base under 10 clicks)")
weak_picks[["rank", "content_id", "clicks_last_30d", "clicks_prev_30d", "action_score"]].head(10)

# Leakage check 1: pre-computed label fields never used as scoring inputs
scoring_inputs = ["clicks_last_30d", "clicks_prev_30d", "avg_position"]
assert "trend_direction" not in scoring_inputs and "trend_pct" not in scoring_inputs

# Leakage check 2: sanity-check agreement with the pre-computed label — NOT used to build the rule,
# only to see how often an independent 30d-window rule lines up with it
agreement_rate = (ranked["action"] == ranked["trend_direction"]).mean()
print(f"Rule vs. pre-computed trend_direction agreement: {agreement_rate:.1%}")

# Leakage check 3: no product/event flags in the scoring inputs
product_flag_cols = [c for c in df.columns if any(k in c.lower() for k in ["redesign", "migrat", "relaunch", "provider_used", "model_used"])]
print(f"Columns excluded from scoring as non-organic signals: {product_flag_cols}")

5216 picks flagged as weak (swing off a base under 10 clicks)
Rule vs. pre-computed trend_direction agreement: 0.0%
Columns excluded from scoring as non-organic signals: ['provider_used', 'model_used']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.